In [1]:
import pandas as pd
import numpy as np
import os, glob
import polars as pl
from tqdm import tqdm
from scipy.fft import rfft, rfftfreq
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score
from sklearn.feature_selection import SelectFromModel
from xgboost import XGBRegressor
from scipy.optimize import minimize
from joblib import Parallel, delayed

def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

In [ ]:
from scipy.signal import butter, lfilter
from scipy.stats import entropy, skew, kurtosis

def butter_lowpass_filter(data, cutoff, fs, order=2):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return lfilter(b, a, data)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

import os

BASE_PATH = '/kaggle/input/competitions/child-mind-institute-problematic-internet-use'
if not os.path.exists(BASE_PATH):
    BASE_PATH = '/home/yopsa_matb/audio_project/data'
SERIES_TRAIN_PATH = f'{BASE_PATH}/series_train.parquet'
SERIES_TEST_PATH = f'{BASE_PATH}/series_test.parquet'

def extract_advanced_audio_features(id_val, folder_type='train'):
    base_series = SERIES_TRAIN_PATH if folder_type == 'train' else SERIES_TEST_PATH
    path = f"{base_series}/id={id_val}/"
    files = glob.glob(os.path.join(path, "*.parquet"))
    if not files: return None
    
    df = pl.read_parquet(files[0])
    
    if df["time_of_day"].dtype == pl.Int64:
        df = df.with_columns((pl.col("time_of_day") / 1e9).alias("seconds_at_day"))
    else:
        df = df.with_columns(
            pl.col("time_of_day").str.to_time().dt.hour().alias("hour"),
            pl.col("time_of_day").str.to_time().dt.second().alias("seconds_at_day")
        )
    
    df = df.with_columns((pl.col("seconds_at_day") // 3600).alias("hour"))
    enmo = df['enmo'].to_numpy()
    enmo_filtered = butter_lowpass_filter(enmo, cutoff=0.2, fs=1.0, order=2)

    wkend_mask = (df["weekday"] >= 5).to_numpy()
    wkday_mask = ~wkend_mask

    restless_mask = (df["hour"] >= 0) & (df["hour"] <= 4)
    restless_enmo = df.filter(restless_mask)["enmo"].to_numpy()
    
    wkend_restless = df.filter(restless_mask & (pl.col("weekday") >= 5))["enmo"].to_numpy()

    enmo_diff = np.diff(enmo_filtered)
    zcr = ((enmo_diff[:-1] * enmo_diff[1:]) < 0).sum() / (len(enmo_diff) + 1)
    
    hourly_means = df.group_by("hour").agg(pl.col("enmo").mean()).sort("hour")["enmo"].to_numpy()
    if len(hourly_means) >= 10:
        sorted_hours = np.sort(hourly_means)
        l5 = np.mean(sorted_hours[:5])
        m10 = np.mean(sorted_hours[-10:])
        ra = (m10 - l5) / (m10 + l5 + 1e-6)
    else:
        l5, m10, ra = 0, 0, 0

    features = {
        'id': id_val,
        'enmo_mean': np.mean(enmo_filtered),
        'enmo_std': np.std(enmo_filtered),
        'zcr_diff': zcr,
        'night_movement_mean': np.mean(restless_enmo) if len(restless_enmo) > 0 else 0,
        'night_movement_std': np.std(restless_enmo) if len(restless_enmo) > 0 else 0,
        'wkend_restless_std': np.std(wkend_restless) if len(wkend_restless) > 0 else 0,
        'wkday_enmo_mean': np.mean(enmo_filtered[wkday_mask]) if wkday_mask.any() else 0,
        'wkend_enmo_mean': np.mean(enmo_filtered[wkend_mask]) if wkend_mask.any() else 0,
        'l5': l5, 'm10': m10, 'ra': ra
    }
        
    return features

In [ ]:
train = pd.read_csv(f'{BASE_PATH}/train.csv')
test = pd.read_csv(f'{BASE_PATH}/test.csv')

train_all = train.copy()
train_labeled = train[train['sii'].notnull()].reset_index(drop=True)
train_unlabeled = train[train['sii'].isnull()].reset_index(drop=True)

print("Extracting features for all training data...")
train_sig = pd.DataFrame([f for f in [extract_advanced_audio_features(i, 'train') for i in tqdm(train_all['id'])] if f])
print("Extracting features for test data...")
test_sig = pd.DataFrame([f for f in [extract_advanced_audio_features(i, 'test') for i in tqdm(test['id'])] if f])

train_enriched = train_all.merge(train_sig, on='id', how='left')
test_enriched = test.merge(test_sig, on='id', how='left')

Extracting features for all training data...


100%|██████████| 3960/3960 [00:31<00:00, 123.82it/s]


Extracting features for test data...


100%|██████████| 20/20 [00:00<00:00, 301.42it/s]


In [ ]:
audio_cols = ['enmo_mean', 'enmo_std', 'zcr_diff', 'night_movement_mean', 'night_movement_std', 'wkend_restless_std', 'wkday_enmo_mean', 'wkend_enmo_mean', 'l5', 'm10', 'ra']

common_cols = [c for c in train.columns if c in test.columns and c != 'id']

missing_threshold = 0.3
null_counts = train[common_cols].isnull().mean()
missing_cols = null_counts[null_counts > missing_threshold].index.tolist()

for col in missing_cols:
    train_enriched[f'{col}_is_null'] = train_enriched[col].isnull().astype(int)
    test_enriched[f'{col}_is_null'] = test_enriched[col].isnull().astype(int)

null_indicators = [f'{col}_is_null' for col in missing_cols]
all_features = common_cols + audio_cols + null_indicators

X_all = train_enriched[all_features].copy()
X_test = test_enriched[all_features].copy()

for col in all_features:
    if not pd.api.types.is_numeric_dtype(X_all[col]):
        X_all[col] = X_all[col].astype(str)
        X_test[col] = X_test[col].astype(str)
        X_all[col], uniques = pd.factorize(X_all[col])
        uniques_list = uniques.tolist()
        X_test[col] = X_test[col].apply(lambda x: uniques_list.index(x) if x in uniques_list else -1)

X_all_final = X_all.values
X_test_final = X_test.values

labeled_mask = train_enriched['sii'].notnull()
X_final = X_all_final[labeled_mask]
y = train_enriched.loc[labeled_mask, 'sii'].astype(int)
X_unlabeled = X_all_final[~labeled_mask]


selector_model = XGBRegressor(n_estimators=100, random_state=42, device='cpu')
selector = SelectFromModel(selector_model, max_features=80, threshold=-np.inf)
selector.fit(X_final, y)
X_final = selector.transform(X_final)
X_test_final = selector.transform(X_test_final)
X_unlabeled = selector.transform(X_unlabeled)

print(f"Final shapes: X_final: {X_final.shape}, y: {y.shape}, X_unlabeled: {X_unlabeled.shape}")

Final shapes: X_final: (2736, 114), y: (2736,), X_unlabeled: (1224, 114)


In [ ]:
class DistributionCalibration:
    def __init__(self):
        self.target_dist = None

    def fit(self, y):
        counts = pd.Series(y).value_counts(normalize=True).sort_index()
        self.target_dist = counts.tolist()

    def predict(self, scores):
        n = len(scores)
        sorted_idx = np.argsort(scores)
        preds = np.zeros(n)
        
        current_idx = 0
        for i, pct in enumerate(self.target_dist):
            count = int(round(pct * n))
            if i == len(self.target_dist) - 1:
                preds[sorted_idx[current_idx:]] = i
            else:
                preds[sorted_idx[current_idx:current_idx + count]] = i
                current_idx += count
        return preds

In [ ]:
import lightgbm as lgb

def train_ensemble(X_tr, y_tr, X_va, y_va, weights):
    xgb_model = XGBRegressor(
        n_estimators=1000, learning_rate=0.01, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, tree_method='hist', early_stopping_rounds=100, device=device
    )
    xgb_model.fit(X_tr, y_tr, sample_weight=weights, eval_set=[(X_va, y_va)], verbose=False)
    
    lgb_model = lgb.LGBMRegressor(
        n_estimators=1000, learning_rate=0.01, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, importance_type='gain', verbose=-1,
        device='gpu' if device == 'cuda' else 'cpu'
    )
    lgb_model.fit(
        X_tr, y_tr, 
        sample_weight=weights, 
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=100)]
    )
    return xgb_model, lgb_model

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("=== Phase 1: Base Ensemble ===")
oof_preds = np.zeros(len(X_final))
test_preds = np.zeros(len(X_test_final))
unlabeled_preds = np.zeros(len(X_unlabeled))

oof_raw = np.zeros(len(X_final))
unlabeled_raw = np.zeros(len(X_unlabeled))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_final, y)):
    X_tr, X_va = X_final[train_idx], X_final[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    weights = np.where(y_tr == 3, 10.0, 1.0)
    
    xgb_m, lgb_m = train_ensemble(X_tr, y_tr, X_va, y_va, weights)
    
    oof_raw[val_idx] = (xgb_m.predict(X_va) + lgb_m.predict(X_va)) / 2
    unlabeled_raw += (xgb_m.predict(X_unlabeled) + lgb_m.predict(X_unlabeled)) / 2 / 5
    test_preds += (xgb_m.predict(X_test_final) + lgb_m.predict(X_test_final)) / 2 / 5
    print(f"Fold {fold+1} complete.")

calibrator = DistributionCalibration()
calibrator.fit(y)
pseudo_labels_raw = calibrator.predict(unlabeled_raw)

confidence_mask = np.abs(unlabeled_raw - pseudo_labels_raw) < 0.2
X_pseudo = X_unlabeled[confidence_mask]
y_pseudo = pseudo_labels_raw[confidence_mask]
print(f"\nUsing {len(X_pseudo)} high-confidence pseudo-labels out of {len(X_unlabeled)}")

print(f"\n=== Phase 2: Refined Training ===")
final_test_preds = np.zeros(len(X_test_final))
final_oof_raw = np.zeros(len(X_final))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_final, y)):
    X_tr_labeled = X_final[train_idx]
    y_tr_labeled = y.iloc[train_idx]
    
    X_tr_comb = np.vstack([X_tr_labeled, X_pseudo])
    y_tr_comb = pd.concat([y_tr_labeled, pd.Series(y_pseudo)])
    
    X_va = X_final[val_idx]
    y_va = y.iloc[val_idx]
    weights = np.where(y_tr_comb == 3, 10.0, 1.0)
    
    xgb_m, lgb_m = train_ensemble(X_tr_comb, y_tr_comb, X_va, y_va, weights)
    
    final_oof_raw[val_idx] = (xgb_m.predict(X_va) + lgb_m.predict(X_va)) / 2
    final_test_preds += (xgb_m.predict(X_test_final) + lgb_m.predict(X_test_final)) / 2 / 5
    print(f"Fold {fold+1} complete.")

calibrator.fit(y)
final_oof_calibrated = calibrator.predict(final_oof_raw)
print(f"\nFinal OOF QWK: {qwk(y, final_oof_calibrated):.4f}")

=== Phase 1: Base Ensemble & Meta-Learner ===
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[489]	valid_0's l2: 0.478829
Fold 1 complete.


/home/yopsa_matb/audio_project/venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/yopsa_matb/audio_project/venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/yopsa_matb/audio_project/venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[364]	valid_0's l2: 0.45843
Fold 2 complete.


/home/yopsa_matb/audio_project/venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/yopsa_matb/audio_project/venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/yopsa_matb/audio_project/venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[319]	valid_0's l2: 0.465377
Fold 3 complete.


...

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[278]	valid_0's l2: 0.484064
Fold 4 complete.


/home/yopsa_matb/audio_project/venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/yopsa_matb/audio_project/venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[195]	valid_0's l2: 0.481467
Fold 5 complete.


/home/yopsa_matb/audio_project/venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/yopsa_matb/audio_project/venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



Final Optimized Thresholds: [0.57377727 0.84358258 2.90871355]
Final Optimized OOF QWK: 0.4507


In [7]:
final_test_preds_calibrated = calibrator.predict(final_test_preds)
submission = pd.DataFrame({
    'id': test['id'],
    'sii': final_test_preds_calibrated.astype(int)
})
submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")
print(submission.head())

Submission saved to submission.csv
         id  sii
0  00008ff9    1
1  000fd460    0
2  00105258    0
3  00115b9f    0
4  0016bb22    1
